In [ ]:
import scanpy as sc
from sklearn.metrics import adjusted_rand_score
import pandas as pd
import os

def preprocess_adata(adata):
    # QC filtering
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)

    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    adata = adata[adata.obs.pct_counts_mt < 5, :]

    # Normalize + log
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    # HVGs
    sc.pp.highly_variable_genes(adata, flavor='seurat', n_top_genes=2000)

    # PCA + neighbors + clustering
    sc.tl.pca(adata, svd_solver='arpack')
    sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
    sc.tl.umap(adata)
    sc.tl.leiden(adata)

    return adata


In [ ]:
!pip3 install igraph

In [ ]:
!pip3 install leidenalg

In [ ]:
adata_gt = sc.read_h5ad("Data/combined_raw.h5ad")
adata_gt = preprocess_adata(adata_gt)


In [ ]:
results = []
missing_fractions = [10, 20, 30]
runs = range(1, 11)

for mf in missing_fractions:  
    for run in runs:
        fname = f"Imputed_KNN_h5ad/adata_dropout_mf{mf}_run{run}_knn_imputed.h5ad"
        if not os.path.exists(fname):
            print(f"Skipping {fname} (not found)")
            continue
        
        adata_imp = sc.read_h5ad(fname)
        adata_imp = preprocess_adata(adata_imp)
        
        # Compute ARI between GT and imputed Leiden clusters
        ari = adjusted_rand_score(adata_gt.obs['leiden'], adata_imp.obs['leiden'])
        
        results.append({
            "method": "MAGIC",
            "missing_fraction": mf/100,  # e.g., 0.1, 0.2, 0.3
            "run": run,
            "ARI": ari  
        })

results_df = pd.DataFrame(results)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Summary table
summary = results_df.groupby(['method','missing_fraction'])['ARI'].agg(['mean','std']).reset_index()
print(summary)

# Barplot with error bars
plt.figure(figsize=(8,6))
sns.barplot(data=summary, x='missing_fraction', y='mean', hue='method', capsize=0.1)

# Add SD error bars manually
for i, row in summary.iterrows():
    plt.errorbar(x=i%3 + (0 if row['method']=='MAGIC' else 0.3),
                 y=row['mean'], yerr=row['std'], fmt='none', c='black', capsize=5)

plt.ylabel("ARI (mean ± SD)")
plt.xlabel("Missing Fraction")
plt.title("Clustering Accuracy after Imputation")
plt.savefig('ARI_Summary_knn.png', dpi=300)
plt.show()


In [ ]:
import scanpy as sc
from sklearn.metrics import adjusted_rand_score
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt

def preprocess_adata(adata):
    # QC filtering
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)

    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    adata = adata[adata.obs.pct_counts_mt < 5, :]

    # Normalize + log
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    # HVGs
    sc.pp.highly_variable_genes(adata, flavor='seurat', n_top_genes=2000)

    # PCA + neighbors + clustering
    sc.tl.pca(adata, svd_solver='arpack')
    sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
    sc.tl.umap(adata)
    sc.tl.leiden(adata)

    return adata


# -----------------------------
# Ground truth clustering
# -----------------------------
adata_gt = sc.read_h5ad("Data/combined_raw.h5ad")
adata_gt = preprocess_adata(adata_gt)
gt_labels = adata_gt.obs['leiden'].copy()

results = []
missing_fractions = [10, 20, 30]
runs = range(1, 11)

# -----------------------------
# Loop over MAGIC-imputed datasets
# -----------------------------
for mf in missing_fractions:  
    for run in runs:
        fname = f"Imputed_h5ad/adata_magic_imputed_mf{mf}_run{run}.h5ad"
        if not os.path.exists(fname):
            print(f"Skipping {fname} (not found)")
            continue
        
        adata_imp = sc.read_h5ad(fname)
        adata_imp = preprocess_adata(adata_imp)
        
        # Make sure cells match between GT and imputed
        common_cells = adata_gt.obs_names.intersection(adata_imp.obs_names)
        gt_sub = gt_labels.loc[common_cells]
        imp_sub = adata_imp.obs['leiden'].loc[common_cells]
        
        # Compute ARI
        ari = adjusted_rand_score(gt_sub, imp_sub)
        
        results.append({
            "method": "MAGIC",   # method label updated
            "missing_fraction": mf/100,  # e.g., 0.1, 0.2, 0.3
            "run": run,
            "ARI": ari  
        })

# -----------------------------
# Results → DataFrame
# -----------------------------
results_df = pd.DataFrame(results)

# Summary table
summary = results_df.groupby(['method','missing_fraction'])['ARI'].agg(['mean','std']).reset_index()
print(summary)

# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(8,6))
sns.barplot(data=summary, x='missing_fraction', y='mean', hue='method', capsize=0.1)

# Add SD error bars manually
for i, row in summary.iterrows():
    plt.errorbar(
        x=i%3 + (0 if row['method']=='MAGIC' else 0.3),
        y=row['mean'], 
        yerr=row['std'], 
        fmt='none', c='black', capsize=5
    )

plt.ylabel("ARI (mean ± SD)")
plt.xlabel("Missing Fraction")
plt.title("Clustering Accuracy after MAGIC Imputation (vs Ground Truth)")
plt.savefig('ARI_Summary_MAGIC.png', dpi=300)
plt.show()


In [ ]:
# Notebook 04: Performance Evaluation & Figures

import scanpy as sc
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score as ari_score

# -----------------------------
# Settings
# -----------------------------
sc.settings.verbosity = 0
sns.set(style="whitegrid")

ground_truth_path = "Data/adata_raw_qc.h5ad"         # ground truth file
imputed_dir = "imputed_iterative"                    # folder with imputed h5ad files
methods = ["IterativeImputer"]                       # method tag for plotting

# -----------------------------
# Preprocessing function
# -----------------------------
def preprocess_adata(adata, n_hvgs=1000):
    """Preprocess AnnData: unique names, HVGs, PCA, clustering."""
    adata = adata.copy()

    # Ensure unique obs/var names
    adata.obs_names_make_unique()
    adata.var_names_make_unique()

    # Handle NaNs/Infs in expression matrix
    X = adata.X.A if hasattr(adata.X, "A") else adata.X
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    adata.X = X

    # Apply log1p only if data not already logged
    if np.max(adata.X) > 30:
        sc.pp.log1p(adata)

    # HVG selection
    sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=n_hvgs)
    adata = adata[:, adata.var["highly_variable"]].copy()

    # PCA + neighbors + clustering
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, svd_solver="arpack")
    sc.pp.neighbors(adata)
    sc.tl.leiden(adata)

    return adata


# -----------------------------
# Load ground truth
# -----------------------------
print(" Preprocessing ground truth...")
adata_gt = sc.read_h5ad(ground_truth_path)
adata_gt = preprocess_adata(adata_gt)

if "leiden" not in adata_gt.obs:
    raise ValueError("Ground truth AnnData does not have 'leiden' clustering.")


# -----------------------------
# Evaluate imputation runs
# -----------------------------
results = []

for fname in os.listdir(imputed_dir):
    if not fname.endswith(".h5ad"):
        continue

    fpath = os.path.join(imputed_dir, fname)
    try:
        adata_imp = sc.read_h5ad(fpath)
        adata_imp = preprocess_adata(adata_imp)

        # Align cells
        common_cells = adata_gt.obs_names.intersection(adata_imp.obs_names)
        adata_gt_sub = adata_gt[common_cells]
        adata_imp_sub = adata_imp[common_cells]

        # Compute ARI
        ari = ari_score(adata_gt_sub.obs["leiden"], adata_imp_sub.obs["leiden"])

        # Parse metadata from filename (mfXX, runX)
        mf = None
        run = None
        for part in fname.split("_"):
            if part.startswith("mf"):
                mf = float(part.replace("mf", "").replace(".h5ad", "")) / 100
            if part.startswith("run"):
                run = int(part.replace("run", "").replace(".h5ad", ""))

        results.append({
            "file": fname,
            "method": "IterativeImputer",
            "ARI": ari,
            "missing_fraction": mf,
            "run": run
        })

        print(f"✅ Processed {fname}: ARI={ari:.3f}")

    except Exception as e:
        print(f"❌ Error with {fname}: {e}")


# -----------------------------
# Results → DataFrame
# -----------------------------
results_df = pd.DataFrame(results)

if results_df.empty:
    raise ValueError("No results computed. Check your imputed files and pipeline.")

results_df = results_df.dropna(subset=["ARI", "missing_fraction"])
results_df["missing_fraction"] = results_df["missing_fraction"].astype(float)

# -----------------------------
# Summary
# -----------------------------
summary = results_df.groupby(["method", "missing_fraction"]).agg(
    ARI_mean=("ARI", "mean"),
    ARI_std=("ARI", "std"),
    runs=("ARI", "count")
).reset_index()

print("\nSummary (mean ± std):")
print(summary)


# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(8, 6))
sns.barplot(
    data=results_df,
    x="missing_fraction",
    y="ARI",
    hue="method",
    errorbar="sd",
    capsize=0.1
)
plt.title("ARI across missing fractions")
plt.ylabel("Adjusted Rand Index")
plt.xlabel("Missing Fraction")
plt.legend(title="Method")
plt.show()


In [ ]:
# Notebook 04: Performance Evaluation & Figures (KNN)

import scanpy as sc
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score as ari_score

# -----------------------------
# Settings
# -----------------------------
sc.settings.verbosity = 0
sns.set(style="whitegrid")

ground_truth_path = "Data/adata_raw_qc.h5ad"     # ground truth file
imputed_dir = "imputed_knn"                      # folder with imputed h5ad files
methods = ["KNNImputer"]                         # method tag for plotting

# -----------------------------
# Preprocessing function
# -----------------------------
def preprocess_adata(adata, n_hvgs=1000):
    """Preprocess AnnData: unique names, HVGs, PCA, clustering."""
    adata = adata.copy()

    # Ensure unique obs/var names
    adata.obs_names_make_unique()
    adata.var_names_make_unique()

    # Handle NaNs/Infs in expression matrix
    X = adata.X.A if hasattr(adata.X, "A") else adata.X
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    adata.X = X

    # Apply log1p only if data not already logged
    if np.max(adata.X) > 30:
        sc.pp.log1p(adata)

    # HVG selection
    sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=n_hvgs)
    adata = adata[:, adata.var["highly_variable"]].copy()

    # PCA + neighbors + clustering
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, svd_solver="arpack")
    sc.pp.neighbors(adata)
    sc.tl.leiden(adata)

    return adata


# -----------------------------
# Load ground truth
# -----------------------------
print(" Preprocessing ground truth...")
adata_gt = sc.read_h5ad(ground_truth_path)
adata_gt = preprocess_adata(adata_gt)

if "leiden" not in adata_gt.obs:
    raise ValueError("Ground truth AnnData does not have 'leiden' clustering.")


# -----------------------------
# Evaluate imputation runs
# -----------------------------
results = []

for fname in os.listdir(imputed_dir):
    if not fname.endswith(".h5ad"):
        continue

    fpath = os.path.join(imputed_dir, fname)
    try:
        adata_imp = sc.read_h5ad(fpath)
        adata_imp = preprocess_adata(adata_imp)

        # Align cells
        common_cells = adata_gt.obs_names.intersection(adata_imp.obs_names)
        adata_gt_sub = adata_gt[common_cells]
        adata_imp_sub = adata_imp[common_cells]

        # Compute ARI
        ari = ari_score(adata_gt_sub.obs["leiden"], adata_imp_sub.obs["leiden"])

        # Parse metadata from filename (mfXX, runX)
        mf = None
        run = None
        for part in fname.split("_"):
            if part.startswith("mf"):
                mf = float(part.replace("mf", "").replace(".h5ad", "")) / 100
            if part.startswith("run"):
                run = int(part.replace("run", "").replace(".h5ad", ""))

        results.append({
            "file": fname,
            "method": "KNNImputer",
            "ARI": ari,
            "missing_fraction": mf,
            "run": run
        })

        print(f"✅ Processed {fname}: ARI={ari:.3f}")

    except Exception as e:
        print(f"❌ Error with {fname}: {e}")


# -----------------------------
# Results → DataFrame
# -----------------------------
results_df = pd.DataFrame(results)

if results_df.empty:
    raise ValueError("No results computed. Check your imputed files and pipeline.")

results_df = results_df.dropna(subset=["ARI", "missing_fraction"])
results_df["missing_fraction"] = results_df["missing_fraction"].astype(float)

# -----------------------------
# Summary
# -----------------------------
summary = results_df.groupby(["method", "missing_fraction"]).agg(
    ARI_mean=("ARI", "mean"),
    ARI_std=("ARI", "std"),
    runs=("ARI", "count")
).reset_index()

print("\nSummary (mean ± std):")
print(summary)


# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(8, 6))
sns.barplot(
    data=results_df,
    x="missing_fraction",
    y="ARI",
    hue="method",
    errorbar="sd",
    capsize=0.1
)
plt.title("ARI across missing fractions (KNNImputer)")
plt.ylabel("Adjusted Rand Index")
plt.xlabel("Missing Fraction")
plt.legend(title="Method")
plt.show()


In [ ]:
# Notebook 04: Performance Evaluation & Figures (SoftImpute)

import scanpy as sc
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score as ari_score

# -----------------------------
# Settings
# -----------------------------
sc.settings.verbosity = 0
sns.set(style="whitegrid")

ground_truth_path = "Data/adata_raw_qc.h5ad"     # ground truth file
imputed_dir = "imputed_softimpute"               # folder with imputed h5ad files
methods = ["SoftImpute"]                         # method tag for plotting

# -----------------------------
# Preprocessing function
# -----------------------------
def preprocess_adata(adata, n_hvgs=1000):
    """Preprocess AnnData: unique names, HVGs, PCA, clustering."""
    adata = adata.copy()

    # Ensure unique obs/var names
    adata.obs_names_make_unique()
    adata.var_names_make_unique()

    # Handle NaNs/Infs in expression matrix
    X = adata.X.A if hasattr(adata.X, "A") else adata.X
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    adata.X = X

    # Apply log1p only if data not already logged
    if np.max(adata.X) > 30:
        sc.pp.log1p(adata)

    # HVG selection
    sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=n_hvgs)
    adata = adata[:, adata.var["highly_variable"]].copy()

    # PCA + neighbors + clustering
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, svd_solver="arpack")
    sc.pp.neighbors(adata)
    sc.tl.leiden(adata)

    return adata


# -----------------------------
# Load ground truth
# -----------------------------
print(" Preprocessing ground truth...")
adata_gt = sc.read_h5ad(ground_truth_path)
adata_gt = preprocess_adata(adata_gt)

if "leiden" not in adata_gt.obs:
    raise ValueError("Ground truth AnnData does not have 'leiden' clustering.")


# -----------------------------
# Evaluate imputation runs
# -----------------------------
results = []

for fname in os.listdir(imputed_dir):
    if not fname.endswith(".h5ad"):
        continue

    fpath = os.path.join(imputed_dir, fname)
    try:
        adata_imp = sc.read_h5ad(fpath)
        adata_imp = preprocess_adata(adata_imp)

        # Align cells
        common_cells = adata_gt.obs_names.intersection(adata_imp.obs_names)
        adata_gt_sub = adata_gt[common_cells]
        adata_imp_sub = adata_imp[common_cells]

        # Compute ARI
        ari = ari_score(adata_gt_sub.obs["leiden"], adata_imp_sub.obs["leiden"])

        # Parse metadata from filename (mfXX, runX)
        mf = None
        run = None
        for part in fname.split("_"):
            if part.startswith("mf"):
                mf = float(part.replace("mf", "").replace(".h5ad", "")) / 100
            if part.startswith("run"):
                run = int(part.replace("run", "").replace(".h5ad", ""))

        results.append({
            "file": fname,
            "method": "SoftImpute",
            "ARI": ari,
            "missing_fraction": mf,
            "run": run
        })

        print(f"✅ Processed {fname}: ARI={ari:.3f}")

    except Exception as e:
        print(f"❌ Error with {fname}: {e}")


# -----------------------------
# Results → DataFrame
# -----------------------------
results_df = pd.DataFrame(results)

if results_df.empty:
    raise ValueError("No results computed. Check your imputed files and pipeline.")

results_df = results_df.dropna(subset=["ARI", "missing_fraction"])
results_df["missing_fraction"] = results_df["missing_fraction"].astype(float)

# -----------------------------
# Summary
# -----------------------------
summary = results_df.groupby(["method", "missing_fraction"]).agg(
    ARI_mean=("ARI", "mean"),
    ARI_std=("ARI", "std"),
    runs=("ARI", "count")
).reset_index()

print("\nSummary (mean ± std):")
print(summary)


# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(8, 6))
sns.barplot(
    data=results_df,
    x="missing_fraction",
    y="ARI",
    hue="method",
    errorbar="sd",
    capsize=0.1
)
plt.title("ARI across missing fractions (SoftImpute)")
plt.ylabel("Adjusted Rand Index")
plt.xlabel("Missing Fraction")
plt.legend(title="Method")
plt.show()


In [ ]:
# Notebook 04: Performance Evaluation & Figures (SoftImpute)

import scanpy as sc
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score as ari_score

# -----------------------------
# Settings
# -----------------------------
sc.settings.verbosity = 0
sns.set(style="whitegrid")

ground_truth_path = "Data/adata_raw_qc.h5ad"     # ground truth file
imputed_dir = "imputed_gan"               # folder with imputed h5ad files
methods = ["SoftImpute"]                         # method tag for plotting

# -----------------------------
# Preprocessing function
# -----------------------------
def preprocess_adata(adata, n_hvgs=1000):
    """Preprocess AnnData: unique names, HVGs, PCA, clustering."""
    adata = adata.copy()

    # Ensure unique obs/var names
    adata.obs_names_make_unique()
    adata.var_names_make_unique()

    # Handle NaNs/Infs in expression matrix
    X = adata.X.A if hasattr(adata.X, "A") else adata.X
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    adata.X = X

    # Apply log1p only if data not already logged
    if np.max(adata.X) > 30:
        sc.pp.log1p(adata)

    # HVG selection
    sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=n_hvgs)
    adata = adata[:, adata.var["highly_variable"]].copy()

    # PCA + neighbors + clustering
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, svd_solver="arpack")
    sc.pp.neighbors(adata)
    sc.tl.leiden(adata)

    return adata


# -----------------------------
# Load ground truth
# -----------------------------
print(" Preprocessing ground truth...")
adata_gt = sc.read_h5ad(ground_truth_path)
adata_gt = preprocess_adata(adata_gt)

if "leiden" not in adata_gt.obs:
    raise ValueError("Ground truth AnnData does not have 'leiden' clustering.")


# -----------------------------
# Evaluate imputation runs
# -----------------------------
results = []

for fname in os.listdir(imputed_dir):
    if not fname.endswith(".h5ad"):
        continue

    fpath = os.path.join(imputed_dir, fname)
    try:
        adata_imp = sc.read_h5ad(fpath)
        adata_imp = preprocess_adata(adata_imp)

        # Align cells
        common_cells = adata_gt.obs_names.intersection(adata_imp.obs_names)
        adata_gt_sub = adata_gt[common_cells]
        adata_imp_sub = adata_imp[common_cells]

        # Compute ARI
        ari = ari_score(adata_gt_sub.obs["leiden"], adata_imp_sub.obs["leiden"])

        # Parse metadata from filename (mfXX, runX)
        mf = None
        run = None
        for part in fname.split("_"):
            if part.startswith("mf"):
                mf = float(part.replace("mf", "").replace(".h5ad", "")) / 100
            if part.startswith("run"):
                run = int(part.replace("run", "").replace(".h5ad", ""))

        results.append({
            "file": fname,
            "method": "GAN",
            "ARI": ari,
            "missing_fraction": mf,
            "run": run
        })

        print(f"✅ Processed {fname}: ARI={ari:.3f}")

    except Exception as e:
        print(f"❌ Error with {fname}: {e}")


# -----------------------------
# Results → DataFrame
# -----------------------------
results_df = pd.DataFrame(results)

if results_df.empty:
    raise ValueError("No results computed. Check your imputed files and pipeline.")

results_df = results_df.dropna(subset=["ARI", "missing_fraction"])
results_df["missing_fraction"] = results_df["missing_fraction"].astype(float)

# -----------------------------
# Summary
# -----------------------------
summary = results_df.groupby(["method", "missing_fraction"]).agg(
    ARI_mean=("ARI", "mean"),
    ARI_std=("ARI", "std"),
    runs=("ARI", "count")
).reset_index()

print("\nSummary (mean ± std):")
print(summary)


# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(8, 6))
sns.barplot(
    data=results_df,
    x="missing_fraction",
    y="ARI",
    hue="method",
    errorbar="sd",
    capsize=0.1
)
plt.title("ARI across missing fractions (GAN)")
plt.ylabel("Adjusted Rand Index")
plt.xlabel("Missing Fraction")
plt.legend(title="Method")
plt.show()


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score as ari_score

# -----------------------------
# Settings
# -----------------------------
sc.settings.verbosity = 0
sns.set(style="whitegrid")

ground_truth_path = "Data/adata_raw_qc.h5ad"   # ground truth file
imputed_root = "imputed_h5ad"   

imputed_dirs = {
    "SoftImpute": "imputed_softimpute",
    "MAGIC": "imputed_h5ad",
    "KNN": "imputed_knn",
    "Mean": "imputed_mean",
    "GAN": "imputed_gan",
    "Iterative": "imputed_iterative"
}               # parent folder with subfolders per method

# Example folder structure:
# imputed_h5ad/
#   ├── gan/
#   ├── knn/
#   ├── softimpute/
#   └── iterative/

# -----------------------------
# Preprocessing function
# -----------------------------
def preprocess_adata(adata, n_hvgs=1000):
    adata = adata.copy()
    adata.obs_names_make_unique()
    adata.var_names_make_unique()

    X = adata.X.A if hasattr(adata.X, "A") else adata.X
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    adata.X = X

    if np.max(adata.X) > 30:
        sc.pp.log1p(adata)

    sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=n_hvgs)
    adata = adata[:, adata.var["highly_variable"]].copy()

    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, svd_solver="arpack")
    sc.pp.neighbors(adata)
    sc.tl.leiden(adata)

    return adata

# -----------------------------
# Load ground truth
# -----------------------------
print(" Preprocessing ground truth...")
adata_gt = sc.read_h5ad(ground_truth_path)
adata_gt = preprocess_adata(adata_gt)

if "leiden" not in adata_gt.obs:
    raise ValueError("Ground truth AnnData does not have 'leiden' clustering.")

# -----------------------------
# Evaluate imputation runs
# -----------------------------
# -----------------------------
# Evaluate imputation runs across all methods
# -----------------------------
results = []

for method, method_dir in imputed_dirs.items():
    if not os.path.isdir(method_dir):
        print(f"⚠️ Skipping {method}, folder not found: {method_dir}")
        continue

    print(f"\n🔹 Processing method: {method}")
    for fname in os.listdir(method_dir):
        if not fname.endswith(".h5ad"):
            continue

        fpath = os.path.join(method_dir, fname)
        try:
            adata_imp = sc.read_h5ad(fpath)
            adata_imp = preprocess_adata(adata_imp)

            # Align cells
            common_cells = adata_gt.obs_names.intersection(adata_imp.obs_names)
            adata_gt_sub = adata_gt[common_cells]
            adata_imp_sub = adata_imp[common_cells]

            # Compute ARI
            ari = ari_score(adata_gt_sub.obs["leiden"], adata_imp_sub.obs["leiden"])

            # Parse metadata from filename
            mf, run = None, None
            for part in fname.split("_"):
                if part.startswith("mf"):
                    mf = float(part.replace("mf", "").replace(".h5ad", "")) / 100
                if part.startswith("run"):
                    run = int(part.replace("run", "").replace(".h5ad", ""))

            results.append({
                "file": fname,
                "method": method,
                "ARI": ari,
                "missing_fraction": mf,
                "run": run
            })

            print(f"✅ {method} | {fname}: ARI={ari:.3f}")

        except Exception as e:
            print(f"❌ Error with {fname}: {e}")

# -----------------------------
# Results → DataFrame
# -----------------------------
results_df = pd.DataFrame(results)
if results_df.empty:
    raise ValueError("No results computed. Check your imputed files and pipeline.")

results_df = results_df.dropna(subset=["ARI", "missing_fraction"])
results_df["missing_fraction"] = results_df["missing_fraction"].astype(float)

# -----------------------------
# Summary
# -----------------------------
summary = results_df.groupby(["method", "missing_fraction"]).agg(
    ARI_mean=("ARI", "mean"),
    ARI_std=("ARI", "std"),
    runs=("ARI", "count")
).reset_index()

print("\nSummary (mean ± std):")
print(summary)

# -----------------------------
# Plot: ARI vs Missing Fraction across Methods
# -----------------------------
plt.figure(figsize=(10, 6))
sns.lineplot(
    data=results_df,
    x="missing_fraction",
    y="ARI",
    hue="method",
    marker="o",
    err_style="band",  # shaded error region
    errorbar="sd"
)
plt.title("ARI across Missing Fractions (All Methods)")
plt.ylabel("Adjusted Rand Index")
plt.xlabel("Missing Fraction")
plt.legend(title="Imputation Method", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()
